anomaly_context.json
        │
        ▼
Load JSON
        │
        ▼
Create Prompt
        │
        ▼
Groq API
        │
        ▼
Root Cause Analysis
        │
        ▼
Recommendations
        │
        ▼
Save Report

# AI-Powered Root Cause Analysis using Groq

This notebook performs automated Root Cause Analysis (RCA) on anomalous logs using a Large Language Model served through the Groq API.

For each detected anomaly, the surrounding context is analysed to determine:

- Probable Root Cause
- Severity
- Impact
- Recommended Fixes
- Confidence Score

The generated report can assist Site Reliability Engineers (SREs) in quickly identifying and resolving production issues.

In [1]:
!pip3 install groq
!pip3 install pandas 



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: /Library/Frameworks/Python.framework/Versions/3.14/bin/python3.14 -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: /Library/Frameworks/Python.framework/Versions/3.14/bin/python3.14 -m pip install --upgrade pip


In [2]:
pip install python-dotenv


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import json
from pathlib import Path

import pandas as pd

from groq import Groq

# Loading Context 

In [4]:
with open("../data/context/anomaly_context.json","r") as f:
    contexts = json.load(f)

print(f"Loaded {len(contexts)} anomaly contexts.")

Loaded 248 anomaly contexts.


In [5]:
from dotenv import load_dotenv
from groq import Groq
import os

load_dotenv(override=True)

client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": "Say SUCCESS"}
    ]
)

print(response.choices[0].message.content)

SUCCESS


In [6]:
import json

def build_prompt(context):

    return f"""
You are an expert Site Reliability Engineer (SRE), DevOps Engineer, and Production Support Specialist.

Analyze the following anomalous log and perform a comprehensive Root Cause Analysis (RCA).

For every issue:
- Identify the most likely root cause.
- Explain why it happened.
- Assess the severity and business impact.
- Suggest immediate troubleshooting steps.
- Suggest permanent fixes.
- Recommend preventive measures to avoid similar incidents in the future.
- Estimate your confidence level.

Return ONLY valid JSON in the following format:

{{
    "RootCause": "...",
    "Reason": "...",
    "Severity": "Critical | High | Medium | Low",
    "Impact": "...",

    "ImmediateFixes": [
        "...",
        "...",
        "..."
    ],

    "PermanentFixes": [
        "...",
        "...",
        "..."
    ],

    "PreventiveMeasures": [
        "...",
        "...",
        "..."
    ],

    "Recommendations": [
        "...",
        "...",
        "..."
    ],

    "Confidence": "95%"
}}

Current Log

{context["CurrentLog"]}

Component

{context["Component"]}

Severity

{context["Severity"]}

Contains Error

{context["ContainsError"]}

Authentication Success

{context["AuthenticationSuccess"]}

Anomaly Score

{context["AnomalyScore"]}

Nearby Logs

{json.dumps(context["NearbyLogs"], indent=4)}

Provide concise, technically accurate, and actionable recommendations suitable for production environments.
"""

In [24]:
rca_results = []

for i, context in enumerate(selected_contexts):

    print(f"Processing {i+1}/{TOP_K}")

    prompt = build_prompt(context)

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        temperature=0,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    rca_results.append({
        "LogIndex": context["LogIndex"],
        "Response": response.choices[0].message.content
    })

Processing 1/20
Processing 2/20
Processing 3/20
Processing 4/20
Processing 5/20
Processing 6/20
Processing 7/20
Processing 8/20
Processing 9/20
Processing 10/20
Processing 11/20
Processing 12/20
Processing 13/20
Processing 14/20
Processing 15/20
Processing 16/20
Processing 17/20
Processing 18/20
Processing 19/20
Processing 20/20


## Parse JSON Responses

The responses returned by the language model are cleaned and converted into structured Python dictionaries.

Any malformed responses are skipped gracefully.

In [25]:
import json
import re

parsed_results = []

for item in rca_results:

    try:

        response = item["Response"]

        response = response.replace("```json","")
        response = response.replace("```","")
        response = response.strip()

        match = re.search(r"\{.*\}", response, re.DOTALL)

        if match:

            result = json.loads(match.group())

            result["LogIndex"] = item["LogIndex"]

            parsed_results.append(result)

        else:

            print(f"No JSON found for Log {item['LogIndex']}")

    except Exception as e:

        print(f"Skipped Log {item['LogIndex']} : {e}")

## Create Structured RCA Report

The parsed JSON responses are converted into a Pandas DataFrame for further analysis and visualization.

In [ ]:
rca_df = pd.DataFrame(parsed_results)

rca_df.head()

In [8]:
preview = pd.DataFrame(selected_contexts)[
    [
        "LogIndex",
        "Component",
        "Severity",
        "AnomalyScore"
    ]
]

preview.head(10)

,LogIndex,Component,Severity,AnomalyScore
0,4526,Globals.cpp,~CR~,3.947018
1,1101,Globals.cpp,~CR~,3.619014
2,4944,Globals.cpp,~CR~,3.566565
3,3605,Globals.cpp,~CR~,3.254536
4,2237,Globals.cpp,~CR~,3.162919
5,4160,Globals.cpp,~CR~,3.132577
6,362,Globals.cpp,~CR~,3.127324
7,454,Globals.cpp,~CR~,3.115484
8,1598,Globals.cpp,~CR~,3.111030
9,2219,Globals.cpp,~CR~,3.103915


In [9]:
rca_results = []

for i, context in enumerate(selected_contexts):

    print(f"Processing {i+1}/{TOP_K}")

    prompt = build_prompt(context)

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        temperature=0,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    rca_results.append({
        "LogIndex": context["LogIndex"],
        "Response": response.choices[0].message.content
    })

Processing 1/20
Processing 2/20
Processing 3/20
Processing 4/20
Processing 5/20
Processing 6/20
Processing 7/20
Processing 8/20
Processing 9/20
Processing 10/20
Processing 11/20
Processing 12/20
Processing 13/20
Processing 14/20
Processing 15/20
Processing 16/20
Processing 17/20
Processing 18/20
Processing 19/20
Processing 20/20


In [23]:
print(rca_results[0]["Response"])

```json
{
    "RootCause": "High TPS (Transactions Per Second) causing system overload",
    "Reason": "The system is experiencing a high volume of transactions per second, leading to potential performance issues and errors. The nearby log '[Globals.cpp|04124|02:46:44:026|~ER~]Received TPS in this second:16' indicates a high TPS, which may be causing the system to become overloaded.",
    "Severity": "High",
    "Impact": "The high TPS may cause delays or failures in message delivery, leading to a negative impact on the business and its customers.",

    "ImmediateFixes": [
        "Monitor the system's performance and adjust the TPS threshold as needed",
        "Implement rate limiting to prevent excessive transactions per second",
        "Verify that the system's resources (e.g., CPU, memory, and network bandwidth) are sufficient to handle the current load"
    ],

    "PermanentFixes": [
        "Optimize the system's configuration and architecture to improve performance and scala

In [10]:
parsed_results = []

for item in rca_results:

    try:
        result = json.loads(item["Response"])

        result["LogIndex"] = item["LogIndex"]

        parsed_results.append(result)

    except Exception as e:
        print(f"Skipped Log {item['LogIndex']}: {e}")

Skipped Log 4526: Expecting value: line 1 column 1 (char 0)
Skipped Log 1101: Expecting value: line 1 column 1 (char 0)
Skipped Log 4944: Expecting value: line 1 column 1 (char 0)
Skipped Log 3605: Expecting value: line 1 column 1 (char 0)
Skipped Log 2237: Expecting value: line 1 column 1 (char 0)
Skipped Log 4160: Expecting value: line 1 column 1 (char 0)
Skipped Log 362: Expecting value: line 1 column 1 (char 0)
Skipped Log 454: Expecting value: line 1 column 1 (char 0)
Skipped Log 1598: Expecting value: line 1 column 1 (char 0)
Skipped Log 2219: Expecting value: line 1 column 1 (char 0)
Skipped Log 1826: Expecting value: line 1 column 1 (char 0)
Skipped Log 4222: Expecting value: line 1 column 1 (char 0)
Skipped Log 1319: Expecting value: line 1 column 1 (char 0)
Skipped Log 2357: Expecting value: line 1 column 1 (char 0)
Skipped Log 342: Expecting value: line 1 column 1 (char 0)
Skipped Log 874: Expecting value: line 1 column 1 (char 0)
Skipped Log 3093: Expecting value: line 1 co

In [17]:
rca_df = pd.DataFrame(parsed_results)

rca_df.head()

""


In [18]:
from pathlib import Path

Path("../outputs").mkdir(exist_ok=True)

rca_df.to_csv(
    "../outputs/root_cause_analysis.csv",
    index=False
)

print("Root Cause Analysis saved successfully.")

Root Cause Analysis saved successfully.


In [22]:
rca_df = pd.DataFrame(parsed_results)
print(len(parsed_results))
print(parsed_results)

0
[]


In [20]:
print("Average Confidence :", rca_df["Confidence"].mode()[0])

print("\nRoot Causes")

display(rca_df["RootCause"].value_counts())

print("\nSeverity Distribution")

display(rca_df["Severity"].value_counts())

KeyError: 'Confidence'